# Week 4 — Comparing two groups

This week answers one question in three forms: **do two groups differ?**

The form depends on what kind of variable you are comparing.

| Outcome | Grouping variable | Test | Effect size |
|---|---|---|---|
| Continuous, approximately normal | Two categories | Independent-samples *t*-test | Cohen's *d* |
| Continuous, not normal (or ordinal) | Two categories | Mann–Whitney *U* | Rank-biserial *r* |
| Categorical | Categorical | Chi-square of independence | Cramér's *V* (φ for 2×2) |

Every analysis follows the same six steps, and you will use them for the rest
of the module:

1. **State** the hypothesis pair.
2. **Inspect** the variables.
3. **Check** the assumptions and decide which test to use.
4. **Run** the test.
5. **Report** the result with an effect size.
6. **Conclude** about the hypothesis.

In [ ]:
import math9102 as m9
import numpy as np
from scipy import stats

m9.use_house_style()

---

## Part 1 — A continuous outcome across two groups

We use the wellbeing survey: 439 respondents from Melbourne, measured on
validated scales for stress, optimism, control and self-esteem.

In [ ]:
survey = m9.load_survey()
survey.shape

### Step 1: State the hypothesis

> **H₀:** There is no difference in perceived stress between respondents who
> have children and respondents who do not.
>
> **H₁:** There is a difference in perceived stress between respondents who have
> children and respondents who do not.

The outcome is `tpstress`, the total score on the Perceived Stress Scale. The
grouping variable is `child`.

### Step 2: Inspect the variables

Start with the grouping variable. How many are in each group, and is anything
missing?

In [ ]:
m9.frequency(survey, "child")

Now the outcome, split by group. Always look at the group sizes and the spread,
not just the means.

In [ ]:
m9.describe_by(survey, "tpstress", "child")

Before going further, check what a complete-case analysis would cost you. Do
this on **the variables you are actually modelling** — not on the whole
dataframe.

In [ ]:
missing = m9.missingness(survey, ["tpstress", "child"])
print(missing.attrs["summary"])
missing

In [ ]:
m9.grouped_box(survey, "tpstress", "child", ylabel="Total Perceived Stress");

### Step 3: Check the assumptions

The *t*-test assumes the outcome is approximately normally distributed
**within each group**. It is a common mistake to check the pooled distribution
instead — that mixes the groups together and can look non-normal simply because
the group means differ.

In [ ]:
for level in ["NO", "YES"]:
    group = survey.loc[survey.child == level, "tpstress"]
    m9.normality_panel(group, label=f"Total Perceived Stress — child = {level}")

In [ ]:
m9.report_normality(survey[survey.child == "NO"], "tpstress",
                    "Total Perceived Stress (no children)")

In [ ]:
m9.report_normality(survey[survey.child == "YES"], "tpstress",
                    "Total Perceived Stress (children)")

Note what the sample size does to the standardised ratios. With several hundred
observations they will flag departures far too small to matter. **The plots
decide the question**; the numbers describe the shape.

#### What about equal variances?

You will meet analyses that run Levene's test and then switch between a
"pooled" and a "separate variances" *t*-test depending on the result. **We do not do
that.** Choosing your test based on the outcome of another test distorts the
error rate of the one that matters, and the decision rule is fragile — a p-value
of .049 and one of .051 lead to different analyses.

Instead we always use **Welch's *t*-test**, which does not assume equal
variances and costs almost nothing when they happen to be equal. Levene's test
is still worth *looking* at, as a description of the data:

In [ ]:
no = survey.loc[(survey.child == "NO") & survey.tpstress.notna(), "tpstress"]
yes = survey.loc[(survey.child == "YES") & survey.tpstress.notna(), "tpstress"]

levene = stats.levene(no, yes, center="median")
print(f"Levene's test:  F = {levene.statistic:.4f},  p = {levene.pvalue:.4f}")
print(f"SD (no children)  = {no.std(ddof=1):.3f}")
print(f"SD (children)     = {yes.std(ddof=1):.3f}")

### Step 4: Run the test

In [ ]:
result = stats.ttest_ind(no, yes, equal_var=False)   # Welch
result

For comparison, the pooled-variance version — the one a Levene-gated
procedure would select:

In [ ]:
pooled = stats.ttest_ind(no, yes, equal_var=True)
print(f"Welch : t = {result.statistic:.4f}, df = {result.df:.1f}, p = {result.pvalue:.4f}")
print(f"Pooled: t = {pooled.statistic:.4f}, df = {pooled.df:.0f}, p = {pooled.pvalue:.4f}")

They agree here. That is the usual case, and it is why defaulting to Welch is
safe.

### Step 5: Effect size

A p-value tells you whether a difference is detectable. It does not tell you
whether it is **large enough to care about**. Always report an effect size.

Cohen's *d* expresses the difference in standard deviations:

In [ ]:
d = m9.cohens_d(no, yes)
print(f"Cohen's d = {d:.3f}  ({m9.interpret(d, 'd')})")

You may see the shortcut *d* = 2*t*/√df in textbooks. It assumes the two groups
are the **same size**, so treat it as an approximation:

In [ ]:
approx = 2 * pooled.statistic / np.sqrt(len(no) + len(yes) - 2)
exact = pooled.statistic * np.sqrt(1 / len(no) + 1 / len(yes))
print(f"exact       : {exact:.4f}")
print(f"2t/sqrt(df) : {approx:.4f}   (n = {len(no)} vs {len(yes)})")

### Step 6: Report and conclude

In [ ]:
m9.report_ttest(
    survey, "tpstress", "child", result,
    outcome_label="Total Perceived Stress",
    group_labels={"NO": "respondents without children",
                  "YES": "respondents with children"},
)

That sentence is **generated** from the data and the result object. Nothing in it
is typed by hand, so it cannot drift from the analysis if the data change.
Work the same way in everything you write.

**Conclusion.** There is not enough evidence to reject the null hypothesis. Note
carefully what this does *not* mean: it does not mean the groups are identical,
and it does not prove H₀. It means this sample does not provide sufficient
evidence of a difference.

---

## Part 2 — When the outcome is not normal

Sometimes the distribution is clearly not suitable for a *t*-test. Here is a
small study of depression scores after two different drinks, measured the
morning after (Sunday) and three days later (Wednesday).

In [ ]:
bdi = m9.load_bdi()
bdi.head()

### Steps 1–2: Hypothesis and inspection

> **H₀:** Depression scores on Wednesday do not differ between the two drinks.
>
> **H₁:** Depression scores on Wednesday differ between the two drinks.

In [ ]:
m9.describe_by(bdi, "bdiwed", "drink")

In [ ]:
m9.grouped_box(bdi, "bdiwed", "drink", ylabel="BDI score (Wednesday)");

### Step 3: Check the assumptions

In [ ]:
for level in sorted(bdi.drink.unique()):
    m9.normality_panel(bdi.loc[bdi.drink == level, "bdiwed"], label=f"BDI Wednesday — {level}")

With ten observations per group there is very little to go on visually, and a
formal normality test has almost no power. When the sample is this small and the
outcome is a bounded questionnaire score, the safer choice is a test that does
not assume normality.

### Steps 4–5: Run the test and get an effect size

The Mann–Whitney *U* test compares the **ranks** of the observations rather than
their means.

In [ ]:
x = bdi.loc[bdi.drink == "DrinkX", "bdiwed"]
y = bdi.loc[bdi.drink == "DrinkY", "bdiwed"]

mw = stats.mannwhitneyu(x, y, alternative="two-sided")
mw

Ties are common in questionnaire data. They are handled by a normal
approximation with a tie correction — they are **not** missing data, and they do
not invalidate the test.

In [ ]:
print(f"tied values present: {bdi.bdiwed.duplicated().any()}")

Two different effect sizes are both conventionally written *r* for this test, so
name whichever you report:

- **Rank-biserial correlation** — computed from *U* directly. Rescaling it as
  (*r* + 1)/2 gives the probability that a randomly chosen member of the first
  group scores above one from the second, which makes it easy to explain.
- **Rosenthal's *r*** = |*z*|/√*N* — the older convention, and still the one
  you will meet in much of the literature.

In [ ]:
print(f"rank-biserial r  = {m9.rank_biserial(x, y):.3f}")
print(f"Rosenthal's r    = {m9.wilcoxon_r(x, y):.3f}")

They answer slightly different questions and do not agree numerically. Pick one,
say which you used, and stay consistent across a report.

### Step 6: Report and conclude

In [ ]:
m9.report_mannwhitney(
    bdi, "bdiwed", "drink", mw,
    outcome_label="depression score on Wednesday",
    group_labels={"DrinkX": "those who drank Drink X",
                  "DrinkY": "those who drank Drink Y"},
)

Now the same comparison on the Sunday measurement, where the picture differs:

In [ ]:
mw_sun = stats.mannwhitneyu(bdi.loc[bdi.drink == "DrinkX", "bdisun"],
                            bdi.loc[bdi.drink == "DrinkY", "bdisun"],
                            alternative="two-sided")

m9.report_mannwhitney(
    bdi, "bdisun", "drink", mw_sun,
    outcome_label="depression score on Sunday",
    group_labels={"DrinkX": "those who drank Drink X",
                  "DrinkY": "those who drank Drink Y"},
)

**Conclusion.** The drinks are indistinguishable the morning after, but differ
three days later. Reporting both is what makes the finding meaningful.

---

## Part 3 — Two categorical variables

When *both* variables are categorical, we ask whether they are **associated**.

In [ ]:
bully = m9.load_bullying()
bully[["ubullsch", "ubulloth"]].head()

### Step 1: State the hypothesis

> **H₀:** Being bullied at school and bullying others are independent.
>
> **H₁:** Being bullied at school and bullying others are associated.

### Step 2: Inspect

In [ ]:
m9.crosstab(bully, "ubullsch", "ubulloth")

Percentages within rows make the comparison readable — each row answers
"of the people in this group, what proportion bullied others?"

In [ ]:
m9.crosstab(bully, "ubullsch", "ubulloth", normalize="index")

### Step 3: Check the assumptions

Chi-square relies on an approximation that needs reasonably large **expected**
counts — conventionally, all above 5.

In [ ]:
table = m9.crosstab(bully.dropna(subset=["ubullsch", "ubulloth"]), "ubullsch", "ubulloth")
chi2, p, dof, expected = stats.chi2_contingency(table, correction=True)

import pandas as pd
pd.DataFrame(expected, index=table.index, columns=table.columns).round(1)

In [ ]:
print(f"smallest expected count: {expected.min():.1f}")

### Steps 4–6: Run, report, conclude

For a 2×2 table we apply Yates' continuity correction.

In [ ]:
m9.report_chisquare(
    table, (chi2, p, dof, expected),
    row_label="having been bullied at school",
    col_label="having bullied others",
)

For a 2×2 table Cramér's *V* is identical to the φ coefficient, which is what
you will see reported in the textbooks for tables of this shape.

**Conclusion.** There is evidence of an association, and its size is small. Both
halves of that sentence matter: with 799 respondents even a weak association
will reach significance, so the effect size is what tells you whether it is
practically important.

---

## Choosing between the tests

In [ ]:
m9.md("""
| Your outcome | Your groups | Use |
|---|---|---|
| Continuous, roughly normal within groups | 2 | Welch's *t*-test |
| Continuous but clearly skewed, or ordinal, or a very small sample | 2 | Mann–Whitney *U* |
| Categorical | Categorical | Chi-square of independence |
""")

Two habits to carry forward:

- **Decide from the data, and say what you decided from.** "The Q-Q plot showed
  the residuals following the reference line closely" is evidence. "The data were
  normal" is not.
- **Never report a p-value on its own.** A significant result with a negligible
  effect size and a non-significant result with a moderate one are both common,
  and both are easy to misread without the effect size beside them.

## Exercise

`exercise.md` asks you to run all three tests on variables you have not seen
yet, and to write the reports yourself.